# Exercise 5 (Spark) [5 points]
---

For this exercise, you will work on this JupyterLab notebook, and solve the tasks listed herein. These tasks, in addition to writing Spark code, require you to analyse various query plans and to reason about them.

To get a deeper understanding, and look up the types and definitions of various functions, we recommend that you visit the Spark and Spark SQL documentation.

### a) From SQL literal syntax to Dataframe (and back again)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel

# Create Spark session
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("StackExchangeAnalysis")
    .getOrCreate()
)

# Define schemas
comments_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("postid", IntegerType(), True),
    StructField("score", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("userdisplayname", StringType(), True),
    StructField("userid", IntegerType(), True)
])

posts_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("acceptedanswerid", StringType(), True),
    StructField("answercount", IntegerType(), True),
    StructField("body", StringType(), True),
    StructField("closeddate", TimestampType(), True),
    StructField("commentcount", IntegerType(), True),
    StructField("communityowneddate", TimestampType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("favoritecount", IntegerType(), True),
    StructField("lastactivitydate", TimestampType(), True),
    StructField("lasteditdate", TimestampType(), True),
    StructField("lasteditordisplayname", StringType(), True),
    StructField("lasteditoruserid", StringType(), True),
    StructField("ownerdisplayname", StringType(), True),
    StructField("owneruserid", IntegerType(), True),
    StructField("parentid", IntegerType(), True),
    StructField("posttypeid", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("tags", StringType(), True),
    StructField("title", StringType(), True),
    StructField("viewcount", IntegerType(), True)
])

users_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("aboutme", StringType(), True),
    StructField("accountid", StringType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("displayname", StringType(), True),
    StructField("downvotes", IntegerType(), True),
    StructField("lastaccessdate", TimestampType(), True),
    StructField("location", StringType(), True),
    StructField("profileimageurl", StringType(), True),
    StructField("reputation", IntegerType(), True),
    StructField("upvotes", IntegerType(), True),
    StructField("views", IntegerType(), True),
    StructField("websiteurl", StringType(), True)
])

votes_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("bountyamount", IntegerType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("postid", IntegerType(), True),
    StructField("userid", IntegerType(), True),
    StructField("votetypeid", IntegerType(), True)
])

# Load CSV files
badgesDF = spark.read.options(header=True, inferSchema=True).csv("/home/adbs_shared/stackexchange/badges.csv")
commentsDF = spark.read.options(header=True, inferSchema=True, multiLine=True).schema(comments_schema).csv("/home/adbs_shared/stackexchange/comments.csv")
postsDF = spark.read.options(header=True, inferSchema=True, multiLine=True).schema(posts_schema).csv("/home/adbs_shared/stackexchange/posts.csv")
postlinksDF = spark.read.options(header=True, inferSchema=True).csv("/home/adbs_shared/stackexchange/postlinks.csv")
usersDF = spark.read.options(header=True, inferSchema=True).schema(users_schema).csv("/home/adbs_shared/stackexchange/users.csv")
votesDF = spark.read.options(header=True, inferSchema=True).schema(votes_schema).csv("/home/adbs_shared/stackexchange/votes.csv")

# Create temporary views for Spark SQL
badgesDF.createOrReplaceTempView("badges")
commentsDF.createOrReplaceTempView("comments")
postsDF.createOrReplaceTempView("posts")
postlinksDF.createOrReplaceTempView("postlinks")
usersDF.createOrReplaceTempView("users")
votesDF.createOrReplaceTempView("votes")

In [ ]:
badgesDF.printSchema()
commentsDF.printSchema()
postsDF.printSchema()
postlinksDF.printSchema()
usersDF.printSchema()
votesDF.printSchema()

#### Query 1: SQL Literal Syntax --> Dataframe API

In [ ]:
query = """
WITH answers AS (
  SELECT p.owneruserid AS user_id, COUNT(v.postid) AS accepted_votes
  FROM posts p
  JOIN votes v ON p.id = v.postid
  WHERE p.posttypeid = '2' AND v.votetypeid = '1'
  GROUP BY p.owneruserid
),
questions AS (
  SELECT p.owneruserid AS user_id, COUNT(v.postid) AS upvotes
  FROM posts p
  JOIN votes v ON p.id = v.postid
  WHERE p.posttypeid = '1' AND v.votetypeid = '2'
  GROUP BY p.owneruserid
)
SELECT a.user_id, a.accepted_votes, q.upvotes
FROM answers a
JOIN questions q ON a.user_id = q.user_id
WHERE a.accepted_votes > q.upvotes;
"""

query1 = spark.sql(query)
query1.show()
# query1.explain()

#### Query 2: Dataframe API --> SQL Literal Syntax  

In [ ]:
user_stats = postsDF.filter(col("owneruserid").isNotNull()).groupBy("owneruserid").agg(
                      count("*").alias("post_count"),
                      sum("score").alias("total_score")
                  )

filtered = user_stats.filter(col("total_score").isNotNull() & 
                             (col("post_count") >= 50))

result = filtered.withColumn(
    "efficiency",
    col("total_score") / col("post_count")
).orderBy(col("efficiency").desc()).limit(50)

result.show()
#result.explain()

### b) SPARK processing model
You are given two independent Spark applications. Each application consists of a sequence of Spark jobs, and each job consists of multiple transformations followed by actions. Two optimization blocks have been introduced in an attempt to improve the performance of these applications.<br>
Based on the provided programs, answer the following questions:
* Ignoring the two optimization blocks, identify the stages and their boundaries for each Spark job: *user_scores*, *tag_scores*, and *user_comment_scores*.
Clearly explain your reasoning using Spark’s processing model.
* For Application 1, analyze and discuss the impact of the two optimization blocks on the performance of the application. Explain whether they improve performance, and justify your answer.
 * For Application 2, analyze and discuss the impact of the two optimization blocks on the performance of the application. Again, explain whether they improve performance, providing justification.


In [ ]:
posts = postsDF.select('id','owneruserid','tags','score').rdd 
posts = posts.filter(lambda x: (x['score'] != None) & (x['owneruserid'] != None) & (x['tags'] != None)).map(lambda x: (x['id'], (x['owneruserid'], x['tags'], int(x['score']))))# (post_id, (user_id, tag, score))

votes = votesDF.rdd
votes = votes.map(lambda x: (x['postid'], 1)) # (post_id, vote_count)

comments = commentsDF.rdd
comments = comments.map(lambda x: (x['postid'], 1)) # (post_id, comment_count)

high_score_posts = posts.filter(lambda x: x[1][2] >= 5)

# ---------------------------
# OPTIMIZATION BLOCK 1
# ---------------------------
high_score_posts = high_score_posts.partitionBy(4).cache()
votes = votes.partitionBy(4)
#-----------------------------

post_votes = high_score_posts.join(votes)
post_comments = high_score_posts.join(comments)

# ---------------------------
# OPTIMIZATION BLOCK 2
# ---------------------------
post_votes.cache()
#-----------------------------

user_scores = post_votes.map(lambda x: (x[1][0][0], x[1][1])) \
                        .reduceByKey(lambda a, b: a + b)

tag_scores = post_votes.map(lambda x: (x[1][0][1], x[1][1])) \
                      .reduceByKey(lambda a, b: a + b)

user_comment_scores = post_comments.map(lambda x: (x[1][0][0], x[1][1])) \
                        .reduceByKey(lambda a, b: a + b)


In [ ]:
# ---------------------------
# Application 1
# ---------------------------
print(user_scores.take(5))
print(tag_scores.take(5))
print(user_comment_scores.take(5))

In [ ]:
# ---------------------------
# Application 2
# ---------------------------
print(user_scores.take(5))
print(tag_scores.take(5))

### c) Translate the algorithm for constructing a Top-K user similarity graph from a user-item interaction graph (algorithm from Exercise 2)

In [ ]:
# Input edges
edges = [
    ("u1", "i1"), ("u1", "i2"), ("u1", "i3"), ("u1", "i4"),
    ("u2", "i1"), ("u2", "i2"), ("u2", "i5"),
    ("u3", "i2"), ("u3", "i3"), ("u3", "i4"),
    ("u4", "i3"), ("u4", "i5")
]

k = 2 

### d) You are given a dataset of retail transactions, where each transaction consists of a list of items bought together. Using Spark RDD transformations and actions, implement an algorithm to identify the top-K most frequently co-purchased item pairs.

In [ ]:
# Input Data
transactions = [
    ["milk", "bread", "eggs", "butter"],
    ["bread", "butter", "jam"],
    ["milk", "bread"],
    ["bread", "eggs"],
    ["milk", "eggs", "yogurt"],
    ["bread", "milk", "butter"],
    ["milk", "bread", "butter", "cheese"],
    ["bread", "eggs", "butter"],
    ["milk", "eggs", "bread"],
    ["bread", "butter", "jam", "honey"],
    ["milk", "yogurt"],
    ["eggs", "bread", "milk", "butter"],
    ["bread", "cheese"],
    ["milk", "bread", "eggs"],
    ["butter", "bread", "jam"],
    ["milk", "bread", "cheese"],
    ["eggs", "milk", "yogurt", "honey"],
    ["bread", "butter", "eggs"],
    ["milk", "bread", "butter", "jam"],
    ["bread", "eggs", "milk"]
]

min_support = 3

---
## **Your solution for Exercise 5 will consist of:**  
*  This notebook, filled with your solutions and discussions/explanations in the report. 
